# Lab 01: Running Your First Benchmark Suite

## Objectives
- Install and configure the LM Evaluation Harness
- Run two benchmark suites (MMLU-Pro and IFEval)
- Parse and compare results across models
- Visualize benchmark performance

## Prerequisites
- Python 3.9+
- OpenAI API key (or access to another LLM provider)
- ~30 minutes for first run

In [ ]:
# Install dependencies
!pip install lm-evaluation-harness==0.4.1 -q
!pip install pandas matplotlib seaborn -q
!pip install openai litellm -q

print("Dependencies installed successfully!")

## Step 1: Configure the Benchmark

Before running evaluations, we need to set up API credentials and define which models we want to test.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Configuration
config = {
    "models": ["gpt-3.5-turbo", "gpt-4-turbo-preview"],
    "benchmarks": ["mmlu_pro", "ifeval"],
    "output_dir": "./benchmark_results",
    "num_shots": 0,  # Zero-shot evaluation
    "batch_size": 4
}

# Create output directory
Path(config["output_dir"]).mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(json.dumps(config, indent=2))

## Step 2: Run MMLU-Pro Subset

MMLU-Pro is a comprehensive benchmark covering multiple domains of knowledge. We'll run a subset for demonstration.

In [ ]:
# Simulated MMLU-Pro evaluation results
# In practice, this would call: lm_eval.evaluate(model, "mmlu_pro", ...)

mmlu_results = {
    "gpt-3.5-turbo": {
        "overall": 0.72,
        "biology": 0.68,
        "chemistry": 0.71,
        "history": 0.75,
        "mathematics": 0.64,
        "physics": 0.70
    },
    "gpt-4-turbo-preview": {
        "overall": 0.87,
        "biology": 0.85,
        "chemistry": 0.89,
        "history": 0.88,
        "mathematics": 0.82,
        "physics": 0.88
    }
}

print("MMLU-Pro Results:")
print(json.dumps(mmlu_results, indent=2))

# Save results
with open(f"{config['output_dir']}/mmlu_pro_results.json", "w") as f:
    json.dump(mmlu_results, f, indent=2)

## Step 3: Run IFEval

IFEval measures instruction-following capability, testing the model's ability to adhere to specific constraints and requirements.

In [ ]:
# Simulated IFEval evaluation results
# In practice, this would call: lm_eval.evaluate(model, "ifeval", ...)

ifeval_results = {
    "gpt-3.5-turbo": {
        "overall_instruction_following": 0.68,
        "constraints_adherence": 0.65,
        "response_format": 0.72,
        "content_requirements": 0.67
    },
    "gpt-4-turbo-preview": {
        "overall_instruction_following": 0.91,
        "constraints_adherence": 0.89,
        "response_format": 0.93,
        "content_requirements": 0.92
    }
}

print("IFEval Results:")
print(json.dumps(ifeval_results, indent=2))

# Save results
with open(f"{config['output_dir']}/ifeval_results.json", "w") as f:
    json.dump(ifeval_results, f, indent=2)

## Step 4: Parse and Compare Results

Load the results and create a structured comparison DataFrame for analysis.

In [ ]:
# Load results
with open(f"{config['output_dir']}/mmlu_pro_results.json") as f:
    mmlu = json.load(f)
    
with open(f"{config['output_dir']}/ifeval_results.json") as f:
    ifeval = json.load(f)

# Create comparison DataFrame
comparison_data = []

for model in config["models"]:
    comparison_data.append({
        "Model": model,
        "MMLU-Pro Overall": mmlu[model]["overall"],
        "IFEval Overall": ifeval[model]["overall_instruction_following"],
        "Average Score": (mmlu[model]["overall"] + ifeval[model]["overall_instruction_following"]) / 2
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n=== Benchmark Comparison ===")
print(df_comparison.to_string(index=False))
print(f"\nGPT-4 Improvement over GPT-3.5:")
print(f"  MMLU-Pro: {(df_comparison.loc[1, 'MMLU-Pro Overall'] - df_comparison.loc[0, 'MMLU-Pro Overall'])*100:.1f}%")
print(f"  IFEval: {(df_comparison.loc[1, 'IFEval Overall'] - df_comparison.loc[0, 'IFEval Overall'])*100:.1f}%")

## Step 5: Visualize Results

Create visual comparisons to better understand model performance across benchmarks.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Overall benchmark scores
benchmark_scores = pd.DataFrame({
    "Model": ["GPT-3.5", "GPT-4"],
    "MMLU-Pro": [mmlu[m]["overall"] for m in config["models"]],
    "IFEval": [ifeval[m]["overall_instruction_following"] for m in config["models"]]
})

x = np.arange(len(benchmark_scores))
width = 0.35
axes[0].bar(x - width/2, benchmark_scores["MMLU-Pro"], width, label="MMLU-Pro", alpha=0.8)
axes[0].bar(x + width/2, benchmark_scores["IFEval"], width, label="IFEval", alpha=0.8)
axes[0].set_ylabel("Score")
axes[0].set_title("Overall Benchmark Comparison")
axes[0].set_xticks(x)
axes[0].set_xticklabels(benchmark_scores["Model"])
axes[0].legend()
axes[0].set_ylim([0, 1])
axes[0].grid(axis="y", alpha=0.3)

# Plot 2: MMLU-Pro domain breakdown
domains = ["Biology", "Chemistry", "History", "Math", "Physics"]
gpt35_scores = [mmlu["gpt-3.5-turbo"][k] for k in ["biology", "chemistry", "history", "mathematics", "physics"]]
gpt4_scores = [mmlu["gpt-4-turbo-preview"][k] for k in ["biology", "chemistry", "history", "mathematics", "physics"]]

x = np.arange(len(domains))
axes[1].bar(x - width/2, gpt35_scores, width, label="GPT-3.5", alpha=0.8)
axes[1].bar(x + width/2, gpt4_scores, width, label="GPT-4", alpha=0.8)
axes[1].set_ylabel("Score")
axes[1].set_title("MMLU-Pro Domain Breakdown")
axes[1].set_xticks(x)
axes[1].set_xticklabels(domains, rotation=45, ha="right")
axes[1].legend()
axes[1].set_ylim([0, 1])
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{config['output_dir']}/benchmark_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nVisualization saved to benchmark_comparison.png")

## Interpretation and Next Steps

### Key Findings
1. **GPT-4 significantly outperforms GPT-3.5** across both benchmarks
2. **Instruction-following** shows the largest gap (23 percentage points on IFEval)
3. **Knowledge benchmarks** show 15 percentage point improvement
4. **Domain variation** in MMLU-Pro suggests different strengths (GPT-4 strongest in chemistry)

### Next Steps
- Lab 02: Build custom LLM-as-judge evaluators for domain-specific tasks
- Lab 03: Evaluate RAG systems end-to-end
- Lab 04: Red-team models for safety vulnerabilities
- Lab 05: Measure agent behavior and trajectory quality

### Best Practices
- Always run multiple seeds/trials for variance estimation
- Monitor API costs when evaluating at scale
- Compare against human baselines when possible
- Document model versions and hyperparameters for reproducibility